# 平均向量作特征(word)

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
import fasttext
import json
import numpy as np

# %%
# 平均向量作特征(word)
model = fasttext.load_model('/home/E22301339/fastText/fastText/cc.en.300.bin')
with open('/home/E22301339/depressionDetection/processedData/total_user_wordpost.json', 'r')as f:
    words = json.load(f)
with open('/home/E22301339/depressionDetection/processedData/total_user_wordpost2.json', 'r')as f:
    words2 = json.load(f)
words = list(words.values())
words2 = list(words2.values())
words.extend(words2)
temp = []
for item in words:
    vectors = []
    for word in item:
        vector = model[word]
        vectors.append(vector)
    if len(vectors) > 0:
        average_vector = np.mean(vectors, axis=0)
        temp.append(average_vector.tolist())
    else:
        # 处理词向量缺失的情况
        print("存在nan")
        temp.append([0] * model.get_dimension())
with open('/home/E22301339/depressionDetection/processedData/average_features.json', 'w')as f:
    json.dump(temp, f)
print("================================")


# 平均向量作特征(sub-emotion)

In [ ]:
# 平均向量作特征(sub-emotion)
import json
import numpy as np

with open('/home/E22301339/depressionDetection/codes-author/data/train2.json', 'r')as f:
    sub_emotions = json.load(f)
with open('/home/E22301339/depressionDetection/codes-author/data/test2.json', 'r')as f:
    sub_emotions2 = json.load(f)
with open('/home/E22301339/depressionDetection/codes-author/emotions_ext/sub-emotions_2.0.json', 'r')as f:
    trust_cluster_vectos = json.load(f)
sub_emotions = list(sub_emotions.values())
sub_emotions2 = list(sub_emotions2.values())
sub_emotions.extend(sub_emotions2)
temp = []
for item in sub_emotions:
    vectors = []
    for sub_emotion in item:
        vector = trust_cluster_vectos[sub_emotion]
        vectors.append(vector)
    if len(vectors) > 0:
        average_vector = np.mean(vectors, axis=0)
        temp.append(average_vector.tolist())
    else:
        # 处理词向量缺失的情况
        print("存在nan")
        temp.append([0] * 300)
with open('/home/E22301339/depressionDetection/processedData/average_features2.json', 'w')as f:
    json.dump(temp, f)
print("================================")

#   TFIDF平均加权词向量作特征

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import json
#   TFIDF平均加权词向量作特征
tfidVector = TfidfVectorizer(smooth_idf=False,sublinear_tf=True)
# tfidVector2 = TfidfVectorizer(smooth_idf=False)
#  读取子情绪序列数据
with open('/home/E22301339/depressionDetection/codes-author/data/train2.json', 'r') as f:
    data = json.load(f)
data = list(data.values())
for i in range(len(data)):
    data[i] = ' '.join(data[i]).strip()
train = data
text = train

with open('/home/E22301339/depressionDetection/codes-author/data/test2.json', 'r') as f:
    data = json.load(f)
data = list(data.values())
for i in range(len(data)):
    data[i] = ' '.join(data[i]).strip()
test = data
text2 = test
text.extend(text2)
# text = ['anger1 positive1 anger2 negative4 negative4', 'anger3 joy1 joy2']

tfidf_matrix = tfidVector.fit_transform(text)
# tfidf_matrix2 = tfidVector2.fit_transform(text2)
# print("使用tfid向量化器实现文本数据提取")
# print(tfidVector.fit_transform(text))
feature_names = tfidVector.get_feature_names_out()
# feature_names2 = tfidVector2.get_feature_names_out()
# print(feature_names)

# 存储每个文档中出现过的词汇的 TF-IDF 值
tfidf_values = []
for i in range(len(text)):
    doc_tfidf = {}
    doc_features = tfidf_matrix[i].nonzero()[1]  # 获取非零元素的索引
    for j in doc_features:
        word = feature_names[j]
        tfidf_value = tfidf_matrix[i, j]
        doc_tfidf[word] = tfidf_value
    tfidf_values.append(doc_tfidf)

# tfidf_values2 = []
# for i in range(len(text2)):
#     doc_tfidf = {}
#     doc_features = tfidf_matrix2[i].nonzero()[1]  # 获取非零元素的索引
#     for j in doc_features:
#         word = feature_names2[j]
#         tfidf_value = tfidf_matrix2[i, j]
#         doc_tfidf[word] = tfidf_value
#     tfidf_values2.append(doc_tfidf)

# tfidf_values.extend(tfidf_values2)
# 打印每个文档中词汇的 TF-IDF 值
# for i, doc_tfidf in enumerate(tfidf_values):
#     print(f"文档 {i+1}:")
#     for word, tfidf in doc_tfidf.items():
#         print(f"词 '{word}' 的 TF-IDF 值为: {tfidf:.4f}")

with open('/home/E22301339/depressionDetection/codes-author/emotions_ext/sub-emotions_2.0.json', 'r')as f:
    trust_cluster_vectos = json.load(f)

vectors = []
for item in tfidf_values:
    vector = []
    for word, tfidf in item.items():
        vector.append(trust_cluster_vectos[word])
    vectors.append(vector)

for i in range(len(tfidf_values)):
    tfidf_values[i] =list(tfidf_values[i].values())
print("================================================")   

tf_list=[]
for i in range(len(tfidf_values)):
    tf_vectors=[0]*300
    for j in range(len(tfidf_values[i])):
            vector=tfidf_values[i][j]*np.array(vectors[i][j])
            vector=vector.tolist()
            if(len(vector)==0):
                vector=[0]*300
            tf_vectors = [tf_vectors[i] + vector[i] for i in range(len(vector))]
    if(len(tfidf_values[i])!=0):
        tf_vectors=(np.array(tf_vectors)/len(tfidf_values[i])).tolist()
    tf_list.append(tf_vectors)
print("=============================================================")
with open('/home/E22301339/depressionDetection/processedData/average_features3.json', 'w')as f:
    json.dump(tf_list, f)

#   SIF加权向量作特征

In [ ]:
import numpy as np
from sklearn.decomposition import TruncatedSVD
def get_weighted_average(We, x, w):
    """
    Compute the weighted average vectors
    :param We: We[i,:] is the vector for word i
    :param x: x[i, :] are the indices of the words in sentence i
    :param w: w[i, :] are the weights for the words in sentence i
    :return: emb[i, :] are the weighted average vector for sentence i
    """
    n_samples = x.shape[0]
    emb = np.zeros((n_samples, We.shape[1]))
    for i in range(n_samples): 
        # emb[i,:] = w[i,:].dot(We[x[i,:],:]) / np.count_nonzero(w[i,:])
        nonzero_indices = np.nonzero(w[i, :])[0]
        emb[i,:] = w[i, nonzero_indices].dot(We[x[i, nonzero_indices], :]) / np.count_nonzero(w[i, :])

    return emb

def compute_pc(X,npc=1):
    """
    Compute the principal components. DO NOT MAKE THE DATA ZERO MEAN!
    :param X: X[i,:] is a data point
    :param npc: number of principal components to remove
    :return: component_[i,:] is the i-th pc
    """
    svd = TruncatedSVD(n_components=npc, n_iter=7, random_state=0)
    svd.fit(X)
    return svd.components_

def remove_pc(X, npc=1):
    """
    Remove the projection on the principal components
    :param X: X[i,:] is a data point
    :param npc: number of principal components to remove
    :return: XX[i, :] is the data point after removing its projection
    """
    pc = compute_pc(X, npc)
    if npc==1:
        XX = X - X.dot(pc.transpose()) * pc
    else:
        XX = X - X.dot(pc.transpose()).dot(pc)
    return XX


def SIF_embedding(We, x, w):
    """
    Compute the scores between pairs of sentences using weighted average + removing the projection on the first principal component
    :param We: We[i,:] is the vector for word i
    :param x: x[i, :] are the indices of the words in the i-th sentence
    :param w: w[i, :] are the weights for the words in the i-th sentence
    :param params.rmpc: if >0, remove the projections of the sentence embeddings to their first principal component
    :return: emb, emb[i, :] is the embedding for sentence i
    """
    emb = get_weighted_average(We, x, w)
    # if  params.rmpc > 0:
        # emb = remove_pc(emb, params.rmpc)
    emb[np.isnan(emb).all(axis=1)] = 0
    emb = remove_pc(emb, 1)
    return emb

if __name__ == '__main__':
    from sklearn.feature_extraction.text import TfidfVectorizer
    import numpy as np
    import json
    #   TFIDF平均加权词向量作特征
    tfidVector = TfidfVectorizer(smooth_idf=False,sublinear_tf=True)
    #  读取子情绪序列数据
    with open('/home/E22301339/depressionDetection/codes-author/data/train2.json', 'r') as f:
        data = json.load(f)
    data = list(data.values())
    for i in range(len(data)):
        data[i] = ' '.join(data[i]).strip()
    train = data
    text = train

    with open('/home/E22301339/depressionDetection/codes-author/data/test2.json', 'r') as f:
        data = json.load(f)
    data = list(data.values())
    for i in range(len(data)):
        data[i] = ' '.join(data[i]).strip()
    test = data
    text2 = test
    text.extend(text2)
    # text = ['anger1 positive1 anger2 negative4 negative4', 'anger3 joy1 joy2']

    tfidf_matrix = tfidVector.fit_transform(text)
    # tfidf_matrix2 = tfidVector2.fit_transform(text2)
    print("使用tfid向量化器实现文本数据提取")
    print(tfidVector.fit_transform(text))
    feature_names = tfidVector.get_feature_names_out()
    # feature_names2 = tfidVector2.get_feature_names_out()
    # print(feature_names)

    # 存储每个文档中出现过的词汇的 TF-IDF 值
    tfidf_values = []
    for i in range(len(text)):
        doc_tfidf = {}
        doc_features = tfidf_matrix[i].nonzero()[1]  # 获取非零元素的索引
        for j in doc_features:
            word = feature_names[j]
            tfidf_value = tfidf_matrix[i, j]
            doc_tfidf[word] = tfidf_value
        tfidf_values.append(doc_tfidf)

    # tfidf_values2 = []
    # for i in range(len(text2)):
    #     doc_tfidf = {}
    #     doc_features = tfidf_matrix2[i].nonzero()[1]  # 获取非零元素的索引
    #     for j in doc_features:
    #         word = feature_names2[j]
    #         tfidf_value = tfidf_matrix2[i, j]
    #         doc_tfidf[word] = tfidf_value
    #     tfidf_values2.append(doc_tfidf)

    # tfidf_values.extend(tfidf_values2)
    # 打印每个文档中词汇的 TF-IDF 值
    # for i, doc_tfidf in enumerate(tfidf_values):
    #     print(f"文档 {i+1}:")
    #     for word, tfidf in doc_tfidf.items():
    #         print(f"词 '{word}' 的 TF-IDF 值为: {tfidf:.4f}")

    with open('/home/E22301339/depressionDetection/codes-author/emotions_ext/sub-emotions_2.0.json', 'r')as f:
        trust_cluster_vectos = json.load(f)

    words=list(trust_cluster_vectos.keys())
    vectors=list(trust_cluster_vectos.values())
    sentence_indices=[]
    for i in range(len(tfidf_values)):
        user_words=list(tfidf_values[i].keys())
        word_indices = [words.index(word) if word in words else -1 for word in user_words]
        sentence_indices.append(word_indices)
    
    max_length = max(len(sentence) for sentence in sentence_indices)  # 获取最大长度
    # 对每个句子进行填充，使其长度与最大长度一致
    padded_sentence_indices = [sentence + [0] * (max_length - len(sentence)) for sentence in sentence_indices]
    
    We=np.array(vectors)
    x=np.array(padded_sentence_indices)
    w=[]
    for i in range(len(tfidf_values)):
        w.append(list(tfidf_values[i].values()))
        
    max_length = max(len(sentence) for sentence in sentence_indices)  # 获取最大长度
    # 对每个句子的 TF-IDF 得分进行填充，使其长度与最大长度一致
    padded_w = [weights + [0.0] * (max_length - len(weights)) for weights in w]

    w=np.array(padded_w)


    result=SIF_embedding(We,x,w)
    result=result.tolist()
    print("======")
    with open('/home/E22301339/depressionDetection/processedData/average_features4.json', 'w')as f:
        json.dump(result, f)


#   SIF加权向量作特征II

In [1]:

if __name__ == '__main__':
    from sklearn.feature_extraction.text import TfidfVectorizer
    import numpy as np
    import json
    import numpy as np
    from sklearn.decomposition import TruncatedSVD
    from SIF import data_io, params,SIF_embedding
    from collections import Counter

    #  读取子情绪序列数据
    with open('/home/E22301339/depressionDetection/codes-author/data/train2.json', 'r') as f:
        data = json.load(f)
    data = list(data.values())
    for i in range(len(data)):
        data[i] = ' '.join(data[i]).strip()
    train = data

    with open('/home/E22301339/depressionDetection/codes-author/data/test2.json', 'r') as f:
        data = json.load(f)
    data = list(data.values())
    for i in range(len(data)):
        data[i] = ' '.join(data[i]).strip()
    test = data
    train.extend(test)

    text=train
    # text=['anger1 anger2 anger3 anger4 anger5','']
    # input
    # wordfile = '../data/glove.840B.300d.txt' # word vector file, can be downloaded from GloVe website
    # weightfile = '../auxiliary_data/enwiki_vocab_min200.txt' # each line is a word and its frequency
    weightpara = 1e-3 # the parameter in the SIF weighting scheme, usually in the range [3e-5, 3e-3]
    rmpc = 1 # number of principal components to remove in SIF weighting scheme
    sentences = text
    merged_text = ' '.join(text)

    # load word vectors
    (words, We) = data_io.getWordmap()
    # load word weights
    word2weight = data_io.getWordWeight(merged_text,weightpara) # word2weight['str'] is the weight for the word 'str'
    weight4ind = data_io.getWeight(words, word2weight) # weight4ind[i] is the weight for the i-th word
    # load sentences
    x, m= data_io.sentences2idx(sentences, words) # x is the array of word indices, m is the binary mask indicating whether there is a word in that location
    w = data_io.seq2weight(x, m, weight4ind) # get word weights

    # set parameters
    params = params.params()
    params.rmpc = rmpc

    # get SIF embedding
    embedding = SIF_embedding.SIF_embedding(We, x, w, params) # embedding[i,:] is the embedding for sentence i
    result=embedding.tolist()
    print("======")
    with open('/home/E22301339/depressionDetection/processedData/average_features4_2.json', 'w')as f:
        json.dump(result, f)


/home/E22301339/depressionDetection/test_files/SIF/SIF_embedding.py:16: RuntimeWarning: invalid value encountered in divide
  emb[i,:] = w[i,:].dot(We[x[i,:],:]) / np.count_nonzero(w[i,:])
